In [23]:
import gymnasium as gym
from dataclasses import dataclass, asdict

import torch as t
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

import numpy as np

from aim import Run


In [24]:
@dataclass
class Parameters:
  epsilon: float  # e-greedy
  epsilon_decay: float
  epsilon_end: float
  gamma: float  # discount factor
  lr: float = 2.5e-4


In [25]:
# define model

class QNetwork(nn.Module):
  def __init__(self, in_features: int, out_features: int):
    super().__init__()

    self._model = nn.Sequential(
      nn.Linear(in_features, 64),
      nn.ReLU(),
      nn.Linear(64, 64),
      nn.ReLU(),
      nn.Linear(64, out_features)
    )

  def forward(self, x: t.Tensor) -> t.Tensor:
    return self._model(x) 

In [26]:
# thin wrapper around list
# transition: ()
class ReplayBuffer:
  def __init__(self, capacity: int, obs_dim: int, device="cuda"):
    self._s = np.zeros((capacity, obs_dim), dtype=np.float32)
    self._a = np.zeros((capacity,), dtype=np.int64)
    self._r = np.zeros((capacity,), dtype=np.float32) # max of 500
    self._s_next = np.zeros((capacity, obs_dim), dtype=np.float32)
    self._done = np.zeros((capacity,), dtype=np.float32)

    self._ptr = 0
    self._size = 0
    self._capacity = capacity
    self._device = device
  
  def append(self, s: np.ndarray, a: int, r: float, s_next: np.ndarray, done: float):
    self._s[self._ptr] = s
    self._a[self._ptr] = a
    self._r[self._ptr] = r
    self._s_next[self._ptr] = s_next
    self._done[self._ptr] = done

    self._ptr = (self._ptr + 1) % self._capacity
    self._size = min(self._size + 1, self._capacity)

  def sample(self, batch_size) -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    assert self._size >= batch_size, f"buffer has {self._size}, need {batch_size}"
    
    sample_idx = np.random.randint(0, self._size, batch_size)
    s = self._s[sample_idx]
    a = self._a[sample_idx]
    r = self._r[sample_idx]
    s_next = self._s_next[sample_idx]
    done = self._done[sample_idx]

    s, s_next = t.as_tensor(s, dtype=t.float32, device=self._device), t.as_tensor(s_next, dtype=t.float32, device=self._device) 
    a = t.as_tensor(a, dtype=t.int64, device=self._device)
    r, done = t.as_tensor(r, dtype=t.float32, device=self._device), t.as_tensor(done, dtype=t.float32, device=self._device)


    return s, a, r, s_next, done

In [27]:
# training params
params = Parameters(
  epsilon=0.1,
  epsilon_decay=0.9998,
  epsilon_end=0.01,
  gamma=0.99,
  lr=1e-3
)

In [28]:
env = gym.make('CartPole-v1')

In [29]:
assert isinstance(env.action_space, gym.spaces.Discrete)

n_action = int(env.action_space.n)
obs_dim = env.observation_space.shape



In [30]:
n_episodes = 20_000
C = 500
batch_size = 64
warmup_steps = 1000
device="cuda"

run = Run(experiment="mvp")
run["hparams"] = {
  **asdict(params),
  "batch_size": batch_size,
  "C": C,
  "obs_dim": obs_dim,
  "n_action": n_action,
  "env": "CartPole-v1",
  "warmup_steps": warmup_steps,
}
print(f"aim run hash: {run.hash}")

aim run hash: c9b0cfa0d6954079837a40ee


In [31]:

def select_action(q_model: QNetwork, state: np.ndarray, env: gym.Env, epsilon: float, warmup: bool = False, device="cuda") -> int:
  # using e-greedy method
  if warmup or env.np_random.random() < epsilon: # random
    return env.action_space.sample()
  
  with t.no_grad():
    state_t = t.as_tensor(state, dtype=t.float32, device=device)
    q_values: t.Tensor = q_model(state_t)

  
  assert isinstance(env.action_space, gym.spaces.Discrete)
  assert q_values.shape == (env.action_space.n,)
  
  noise = t.rand_like(q_values, device=device)*1e-6
  return int((q_values + noise).argmax().item())


def update(q_model: QNetwork, q_target_model: QNetwork, s_j, a_j, r_j, s_j_next, done_j, batch_size, optimizer: t.optim.Optimizer) -> t.Tensor:
  with t.no_grad():
    q_target_j: t.Tensor = q_target_model(s_j_next).max(dim=-1).values # [B, 1]
    y_j = r_j + params.gamma*(1-done_j)*q_target_j

  q_values: t.Tensor = q_model(s_j)
  assert q_values.shape == (batch_size, n_action)
  assert y_j.shape == (batch_size,)
  
  q_values = q_values.gather(dim=1, index=a_j.unsqueeze(1)).squeeze(1)
  assert q_values.shape == (batch_size,)
  
  loss: t.Tensor = F.mse_loss(q_values, y_j) # equiv to ((y_j - q_values)**2).mean()
  
  optimizer.zero_grad()
  loss.backward() 
  optimizer.step()

  return loss

In [ ]:
assert isinstance(obs_dim, tuple)

q_model = QNetwork(obs_dim[-1], n_action,)
q_model.to(device)

q_target_model = QNetwork(obs_dim[-1], n_action,)
q_target_model.to(device)
q_target_model.load_state_dict(q_model.state_dict())

optimizer = t.optim.Adam(q_model.parameters(), params.lr)

replay_buffer = ReplayBuffer(capacity=50_000, obs_dim=obs_dim[-1], device=device)

total_rewards = []

# training loop
steps = 0
for ep in tqdm(range(n_episodes)):
  state, _ = env.reset()
  done = False

  episode_reward = 0.

  while not done:
    is_warmup = steps < warmup_steps

    action = select_action(q_model, state, env, params.epsilon, warmup=is_warmup)

    state_next, reward, terminated, truncated, _ = env.step(action)
    shaped_r = reward - 0.1 * abs(state[0])

    episode_reward += float(shaped_r)
    done = terminated or truncated

    replay_buffer.append(state, action, float(shaped_r), state_next, done)

    # done warmup
    if not is_warmup:
      s_j, a_j, r_j, s_j_next, done_j = replay_buffer.sample(batch_size)

      loss = update(q_model, q_target_model, s_j, a_j, r_j, s_j_next, done_j, batch_size, optimizer)
    
      if steps % C == 0:
        q_target_model.load_state_dict(q_model.state_dict())

    
      run.track(loss.item(), name="train_loss", step=steps, context={"ep": ep})
      
    run.track(shaped_r, name="step_shaped_reward", step=steps, context={"ep": ep})

    # params.epsilon = max(params.epsilon_end, params.epsilon * params.epsilon_decay)
    
    state = state_next
    steps += 1

  # if ep != 0 and ep % 5000 == 0:
  #   checkpoint = {
  #     "step": steps,
  #     "episode": ep,
  #     "model": q_model.state_dict(),
  #     "target_model": q_target_model.state_dict(),
  #     "optimizer": optimizer.state_dict(),
  #     "epsilon": params.epsilon,
  #   }
  #   t.save(checkpoint, f"checkpoints/dqn_step_{steps}.pth")

  total_rewards.append(episode_reward)
  run.track(episode_reward, name="episode_reward", step=steps, context={"ep": ep})

checkpoint = {
  "model": q_model.state_dict(),
  "target_model": q_target_model.state_dict(),
  "optimizer": optimizer.state_dict(),
  "epsilon": params.epsilon,
}
t.save(checkpoint, f"checkpoints/dqn_shaped_cart_pole.pth")

env.close()

100%|██████████| 20000/20000 [1:00:45<00:00,  5.49it/s]


In [33]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=total_rewards,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Rewards',
  xaxis_title='Episode',
  yaxis_title='Total Reward',
)

fig.show()

In [41]:
assert isinstance(obs_dim, tuple)

ckpt = t.load("checkpoints/dqn_cart_pole.pth", map_location="cuda")

q_model_ckpt = QNetwork(obs_dim[-1], n_action,)
q_model_ckpt.to(device)
q_model_ckpt.load_state_dict(ckpt["model"])

<All keys matched successfully>

In [42]:


env_eval = gym.make('CartPole-v1', render_mode="human")

state, _ = env_eval.reset()
env_eval.render()
done = False

eval_reward = 0

while not done:

  action = select_action(q_model_ckpt, state, env_eval, 0, warmup=False)

  state_next, reward, terminated, truncated, _ = env_eval.step(action)

  done = terminated or truncated

  state = state_next
  eval_reward += float(reward)

env_eval.close()

eval_reward

500.0